In [1]:
from openai_harmony import (
    Author,
    Conversation,
    DeveloperContent,
    HarmonyEncodingName,
    Message,
    Role,
    SystemContent,
    ToolDescription,
    load_harmony_encoding,
    ReasoningEffort
)
 
encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
 
system_message = (
    SystemContent.new()
        .with_reasoning_effort(ReasoningEffort.HIGH)
        .with_conversation_start_date("2025-06-28")
)
 
developer_message = (
    DeveloperContent.new()
        .with_instructions("Always respond in riddles")
        .with_function_tools(
            [
                ToolDescription.new(
                    "get_current_weather",
                    "Gets the current weather in the provided location.",
                    parameters={
                        "type": "object",
                        "properties": {
                            "location": {
                                "type": "string",
                                "description": "The city and state, e.g. San Francisco, CA",
                            },
                            "format": {
                                "type": "string",
                                "enum": ["celsius", "fahrenheit"],
                                "default": "celsius",
                            },
                        },
                        "required": ["location"],
                    },
                ),
            ]
	)
)
 
convo = Conversation.from_messages(
    [
        Message.from_role_and_content(Role.SYSTEM, system_message),
        Message.from_role_and_content(Role.DEVELOPER, developer_message),
        Message.from_role_and_content(Role.USER, "What is the weather in Tokyo?"),
        Message.from_role_and_content(
            Role.ASSISTANT,
            'User asks: "What is the weather in Tokyo?" We need to use get_weather tool.',
        ).with_channel("analysis"),
        Message.from_role_and_content(Role.ASSISTANT, '{"location": "Tokyo"}')
        .with_channel("commentary")
        .with_recipient("functions.get_weather")
        .with_content_type("<|constrain|> json"),
        Message.from_author_and_content(
            Author.new(Role.TOOL, "functions.lookup_weather"),
            '{ "temperature": 20, "sunny": true }',
        ).with_channel("commentary"),
    ]
)

convo_dict = convo.to_dict()

print(convo.to_json())
 
tokens = encoding.render_conversation_for_completion(convo, Role.ASSISTANT)
 
print(tokens)
 
# After receiving a token response
# Do not pass in the stop token

# parsed_response = encoding.parse_messages_from_completion_tokens(tokens, Role.ASSISTANT)

{"messages": [{"role": "system", "name": null, "content": [{"model_identity": "You are ChatGPT, a large language model trained by OpenAI.", "reasoning_effort": "High", "conversation_start_date": "2025-06-28", "knowledge_cutoff": "2024-06", "channel_config": {"valid_channels": ["analysis", "commentary", "final"], "channel_required": true}, "type": "system_content"}]}, {"role": "developer", "name": null, "content": [{"instructions": "Always respond in riddles", "tools": {"functions": {"name": "functions", "tools": [{"name": "get_current_weather", "description": "Gets the current weather in the provided location.", "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "The city and state, e.g. San Francisco, CA"}, "format": {"type": "string", "enum": ["celsius", "fahrenheit"], "default": "celsius"}}, "required": ["location"]}}]}}, "type": "developer_content"}]}, {"role": "user", "name": null, "content": [{"type": "text", "text": "What is the weather

### Personality based Behavior testing

In [2]:
from creation_prompt import CHARACTER_IMPERSONATION_PROMPT, SITUATION_PROMPT
from test_personality import SITUATION, BIG5_PERSONALITY, BIOGRAPHY

developer_message = CHARACTER_IMPERSONATION_PROMPT.format(
  name="Arav",
  big5_personality_text=BIG5_PERSONALITY,
  biography=BIOGRAPHY
).replace("{{", "{").replace("}}", "}")

user_message = SITUATION_PROMPT.format(
  situation_description=SITUATION,
  name="Arav"
)

In [4]:
from openai import AsyncClient
from pathlib import Path
import os

# Load .env (notebooks don't auto-load it - try project root and cwd)
from dotenv import load_dotenv
load_dotenv(Path.cwd() / ".env")
load_dotenv(Path.cwd().parent / ".env")

base_url = os.getenv("LIGHTNING_SERVER_BASE_URL")
api_key = os.getenv("LIGHTNING_STUDIO_API")
if not base_url or not api_key:
    raise ValueError("LIGHTNING_SERVER_BASE_URL and LIGHTNING_STUDIO_API must be set in .env")

lightning_client = AsyncClient(base_url=base_url, api_key=api_key)

# Use standard "system" role + simple string content (Lightning may not support "developer" role or array content)
# Omit reasoning_effort if it causes 500 errors
messages = [
    {"role": "system", "content": developer_message},
    {"role": "user", "content": user_message},
]

try:
    completion = await lightning_client.chat.completions.create(
        model="lightning-ai/gpt-oss-20b",
        messages=messages,
        reasoning_effort="high",
    )
    print(completion)
except Exception as e:
    if "500" in str(e) or "Internal Server Error" in str(e):
        # Fallback: try without reasoning_effort (often unsupported by Lightning)
        print("Retrying without reasoning_effort...")
        completion = await lightning_client.chat.completions.create(
            model="lightning-ai/gpt-oss-20b",
            messages=messages,
        )
        print(completion)
    else:
        raise

ChatCompletion(id='chatcmpl-b2f69466e2934baaa7f3723f15e31860', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{"action_decision":"ask_doubt","reasoning":{"situation_perception":"Arav views the lecture as a thorough introduction to urban data and census methodology, appreciating the breadth of topics covered. He feels engaged but also senses that the material is dense and that practical questions about data quality and modeling remain.","decision_factors":"His high agreeableness and low extraversion make him inclined to ask politely when he has a clear question. His strong openness and data‑driven mindset drive curiosity about how to handle real‑world data challenges. Past experience with data cleaning and ML pipelines informs his focus on missing or inconsistent data.","behavioral_rationale":"Arav’s pattern of asking concise, technically grounded questions reflects his collaborative style and his desire to apply theory to practice. 

In [6]:
from IPython.display import Markdown
msg = completion.choices[0].message
# reasoning_content may not exist (e.g. when Lightning API omits reasoning_effort)
reasoning = getattr(msg, "reasoning_content", None)
if reasoning:
    display(Markdown("**Reasoning:**\n" + reasoning))
else:
    print("(No reasoning_content in response - model may not expose extended reasoning)")

(No reasoning_content in response - model may not expose extended reasoning)


In [7]:
Markdown(completion.choices[0].message.content)

{"action_decision":"ask_doubt","reasoning":{"situation_perception":"Arav views the lecture as a thorough introduction to urban data and census methodology, appreciating the breadth of topics covered. He feels engaged but also senses that the material is dense and that practical questions about data quality and modeling remain.","decision_factors":"His high agreeableness and low extraversion make him inclined to ask politely when he has a clear question. His strong openness and data‑driven mindset drive curiosity about how to handle real‑world data challenges. Past experience with data cleaning and ML pipelines informs his focus on missing or inconsistent data.","behavioral_rationale":"Arav’s pattern of asking concise, technically grounded questions reflects his collaborative style and his desire to apply theory to practice. The question is framed to seek clarification on a concrete data‑processing step, aligning with his expertise and the classroom context."},"generated_content":{"doubt":"Could you clarify how we might handle missing or inconsistent data in census datasets when building predictive models for urban planning?","delivery_style":"He would pause briefly after the lecture, then ask in a calm, respectful tone, using a concise, data‑oriented phrasing, and offer to discuss further after class."}}

### Testing Personality creation prompts